# Kaggle 2× T4 — Ternary Bonsai 2 27B PQ2_0 + PrismML llama.cpp

This notebook runs the **official Ternary-Bonsai-2-27B-PQ2_0.gguf** through the **PrismML llama.cpp fork**.

### Optimized configuration

- Exact PQ2_0 GGUF (~7.21 GB)
- PrismML's official `Bonsai-demo` setup
- CUDA / `-ngl 99`
- Flash attention
- **262,144-token maximum model context**
- **4-bit KV cache** for the full-context configuration
- **Single unified server slot (`-np 1`)** for one-user prompt-cache reuse
- **24 GB RAM prompt cache** for long conversations
- Proper `llama-bench` diagnostics
- GPU telemetry during generation
- 32K / 64K / 131K / 262K context notes and testing
- OpenAI-compatible `llama-server`
- Optional Cloudflare tunnel

> **Why PQ2_0?** PrismML's current model card lists PQ2_0 at 7.21 GB and says it is the faster prompt-processing packing. Bonsai 2 supports up to 262,144 tokens. The official runtime is the PrismML llama.cpp fork; stock llama.cpp/Ollama are not the supported runtime. citeturn0search9turn0search2

> **Why KV4?** `BONSAI_KV4=1` changes the KV cache to Q4_0 and cuts KV memory roughly 3.5×. PrismML describes it as a long-context memory optimization, not a speed optimization; decode can be slightly slower. citeturn0search6turn0search1

> **Why `-np 1`?** For a single user/conversation, PrismML documents that one slot avoids prompt-cache fragmentation/re-prefill. A larger `--cache-ram` can retain long prefixes in RAM. citeturn1search6


## 1. Check the GPU

Run this first. You should see **two Tesla T4 GPUs**.

In [ ]:
!nvidia-smi

## 2. Install the PrismML Bonsai runtime

We use the official `Bonsai-demo` setup path. Open WebUI and the code interpreter are disabled to keep the Kaggle environment smaller.


In [ ]:
import os
import subprocess
from pathlib import Path

DEMO_DIR = Path("/kaggle/working/Bonsai-demo")
MODEL_DIR = Path("/kaggle/working/bonsai-model")
MODEL_FILE = "Ternary-Bonsai-2-27B-PQ2_0.gguf"
MODEL_PATH = MODEL_DIR / MODEL_FILE

MODEL_DIR.mkdir(parents=True, exist_ok=True)

if not DEMO_DIR.exists():
    subprocess.run(
        [
            "git", "clone", "--depth", "1",
            "https://github.com/PrismML-Eng/Bonsai-demo.git",
            str(DEMO_DIR),
        ],
        check=True,
    )

setup_env = os.environ.copy()
setup_env.update({
    "BONSAI_MODEL": "27B",
    "BONSAI_FAMILY": "bonsai2",
    "BONSAI_OPENWEBUI": "0",
    "BONSAI_CODE_INTERPRETER": "0",
    "BONSAI_SKIP_GGUF": "1",
    "BONSAI_SKIP_MLX": "1",
})

subprocess.run(
    ["bash", "-lc", "./setup.sh"],
    cwd=str(DEMO_DIR),
    env=setup_env,
    check=True,
)

print("Demo:", DEMO_DIR)
print("✅ PrismML runtime setup complete")


## 3. Download exactly the Bonsai 2 PQ2_0 GGUF

This is the **7.21 GB** PQ2_0 file from the official PrismML repository, rather than the 53.8 GB FP16 file.


In [ ]:
import subprocess

subprocess.run(
    ["pip", "install", "-q", "-U", "huggingface_hub"],
    check=True,
)

from huggingface_hub import hf_hub_download

if not MODEL_PATH.exists():
    print("⬇️ Downloading Ternary-Bonsai-2-27B-PQ2_0.gguf (~7.21 GB)...")
    hf_hub_download(
        repo_id="prism-ml/Ternary-Bonsai-2-27B-gguf",
        filename=MODEL_FILE,
        local_dir=str(MODEL_DIR),
    )
else:
    print("✅ PQ2_0 model already exists")

size_gb = MODEL_PATH.stat().st_size / (1024 ** 3)
print(f"✅ Model size: {size_gb:.2f} GB")
print(f"✅ Model path: {MODEL_PATH}")


## 4. Configure the supported PrismML llama-server

The official launcher uses `llama-server` on port 8080 with CUDA offload, flash attention and the Bonsai 2 sampling profile.


In [ ]:
import os
import shutil
import subprocess
import time
import urllib.request
from pathlib import Path

LLAMA_URL = "http://127.0.0.1:8080"

# Compatibility aliases for code that used the old Ollama variable names.
OLLAMA_URL = LLAMA_URL

SOURCE_MODEL = str(MODEL_PATH)
MTP_MODEL = "ternary-bonsai-2-27b-pq2"

# Bonsai 2's actual maximum context.
NUM_CTX = 262144

# Bonsai 2 has no official DSpark drafter yet.
DRAFT_TOKENS = 0

# PrismML long-context options.
BONSAI_KV4 = "1"
BONSAI_NGL = "99"

# Single-user server: one unified slot + large RAM prefix cache.
LLAMA_NP = "1"
LLAMA_CACHE_RAM = "24576"

os.environ.update({
    "BONSAI_GGUF": SOURCE_MODEL,
    "BONSAI_CTX": str(NUM_CTX),
    "BONSAI_NGL": BONSAI_NGL,
    "BONSAI_KV4": BONSAI_KV4,
    "BONSAI_HOST": "127.0.0.1",
    "PORT": "8080",
})

def find_binary(name):
    candidates = [
        DEMO_DIR / "bin" / "cuda" / name,
        DEMO_DIR / "llama.cpp" / "build" / "bin" / name,
        DEMO_DIR / "llama.cpp" / "build-cuda" / "bin" / name,
    ]
    for p in candidates:
        if p.exists():
            return str(p)

    found = shutil.which(name)
    if found:
        return found

    matches = list(DEMO_DIR.rglob(name))
    return str(matches[0]) if matches else None

LLAMA_SERVER_BIN = find_binary("llama-server")
LLAMA_CLI_BIN = find_binary("llama-cli")
LLAMA_BENCH_BIN = find_binary("llama-bench")

if not LLAMA_SERVER_BIN:
    raise RuntimeError(
        "PrismML llama-server was not found. Re-run the PrismML setup cell."
    )

print("✅ PrismML llama-server:", LLAMA_SERVER_BIN)
print("✅ PrismML llama-cli:", LLAMA_CLI_BIN)
print("✅ PrismML llama-bench:", LLAMA_BENCH_BIN)
print("✅ Model:", SOURCE_MODEL)
print("✅ Context:", NUM_CTX)
print("✅ KV4:", BONSAI_KV4)
print("✅ GPU layers:", BONSAI_NGL)
print("✅ Server slots:", LLAMA_NP)
print("✅ RAM prompt cache:", f"{LLAMA_CACHE_RAM} MB")


## 5. Start the optimized PrismML llama-server

This uses the official launcher with:

- `BONSAI_CTX=262144`
- `BONSAI_KV4=1`
- `-ngl 99`
- flash attention (enabled by the PrismML launcher)
- `-np 1` for a single conversation
- `--cache-ram 24576` for a large reusable prompt cache

PrismML documents 262K as the Bonsai 2 maximum context and `BONSAI_KV4=1` as the long-context memory option. citeturn0search0turn0search6


In [ ]:
def server_ready():
    try:
        with urllib.request.urlopen(f"{LLAMA_URL}/health", timeout=2) as r:
            return r.status == 200
    except Exception:
        return False

if not server_ready():
    server_log = open("/tmp/bonsai-llama-server.log", "a")

    llama_env = os.environ.copy()
    llama_env.update({
        "BONSAI_GGUF": str(MODEL_PATH),
        "BONSAI_CTX": str(NUM_CTX),
        "BONSAI_NGL": BONSAI_NGL,
        "BONSAI_KV4": BONSAI_KV4,
        "BONSAI_HOST": "127.0.0.1",
        "PORT": "8080",
    })

    # Official PrismML launcher.
    # Extra args are passed through to llama-server.
    llama_server_process = subprocess.Popen(
        [
            "./scripts/start_llama_server.sh",
            "-np", LLAMA_NP,
            "--cache-ram", LLAMA_CACHE_RAM,
        ],
        cwd=str(DEMO_DIR),
        env=llama_env,
        stdout=server_log,
        stderr=subprocess.STDOUT,
    )

    # This notebook is rendered and run unattended by kaggle-rotate, so a cold
    # 262K-context load has to be given room. The interactive 180s cap raced that and
    # aborted boots that were about to succeed. Exiting early on a dead process keeps a
    # genuine crash from waiting out the full budget.
    for _ in range(900):
        if server_ready():
            break
        if llama_server_process.poll() is not None:
            break
        time.sleep(1)

    if not server_ready():
        server_log.close()
        raise RuntimeError(
            "PrismML llama-server did not become ready. "
            "Check /tmp/bonsai-llama-server.log"
        )

print("✅ PrismML llama-server is ready:", LLAMA_URL)
print("✅ 262K context enabled")
print("✅ KV4 enabled")
print("✅ Single-user slot:", LLAMA_NP)
print("✅ RAM prompt cache:", f"{LLAMA_CACHE_RAM} MB")


## 6. Verify the Bonsai 2 PQ2_0 model

In [ ]:
import json
import urllib.request

payload = {
    "model": MTP_MODEL,
    "messages": [
        {"role": "user", "content": "Reply with exactly: Model is working."}
    ],
    "stream": False,
    "temperature": 1.0,
    "top_p": 0.95,
    "top_k": 20,
    "min_p": 0.05,
    "max_tokens": 256,
    "reasoning_effort": "medium",
}

request = urllib.request.Request(
    f"{LLAMA_URL}/v1/chat/completions",
    data=json.dumps(payload).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    method="POST",
)

with urllib.request.urlopen(request, timeout=600) as response:
    result = json.loads(response.read())

print(result["choices"][0]["message"]["content"].strip())
print("✅ Bonsai 2 PQ2_0 is working through PrismML llama.cpp.")


## 9. Raw llama.cpp benchmark

This separates **raw model throughput** from HTTP/API overhead. PrismML's community benchmark uses `llama-bench` with `-ngl 99`, flash attention, and `-p 512 -n 128`. Current community results on RTX 5080/5090 are much faster than T4-class hardware, so this is the cleanest way to determine whether your ~20 tok/s result is a hardware/runtime limit. citeturn1search0turn1search2


In [ ]:
if not LLAMA_BENCH_BIN:
    print("❌ llama-bench was not found.")
else:
    print("Running PQ2_0 CUDA benchmark: prompt 512, generation 128, 3 repetitions.")
    subprocess.run(
        [
            LLAMA_BENCH_BIN,
            "-m", str(MODEL_PATH),
            "-p", "512",
            "-n", "128",
            "-ngl", "99",
            "-fa", "1",
            "-r", "3",
        ],
        check=False,
    )


## 10. API benchmark — thinking OFF vs MEDIUM

The earlier ~19.9 tok/s measurement includes the model's reasoning behavior. This benchmark separates raw non-thinking generation from the normal reasoning preset.

Use the **OFF** result to compare hardware/backend decode speed. Use **MEDIUM** for normal Bonsai 2 reasoning behavior.


In [ ]:

import json
import time
import urllib.request
import urllib.error

prompt = """Explain in about 200 words why GPUs are useful for running large language models.
Cover memory bandwidth, parallel computation, and matrix multiplication."""

# Use the exact model ID reported by PrismML llama-server.
MODEL = "/kaggle/working/bonsai-model/Ternary-Bonsai-2-27B-PQ2_0.gguf"

def run_api():
    payload = {
        "model": MODEL,
        "messages": [
            {"role": "user", "content": prompt}
        ],
        "stream": False,

        # Sampling
        "temperature": 1.0,
        "top_p": 0.95,
        "top_k": 20,
        "min_p": 0.05,

        # Give the model enough room for reasoning + final answer.
        "max_tokens": 1024,
    }

    start = time.perf_counter()

    request = urllib.request.Request(
        f"{LLAMA_URL}/v1/chat/completions",
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST",
    )

    try:
        with urllib.request.urlopen(request, timeout=1200) as response:
            result = json.loads(response.read())

    except urllib.error.HTTPError as e:
        print(f"HTTP {e.code}: {e.reason}")
        print(e.read().decode("utf-8", errors="replace"))
        return None

    wall = time.perf_counter() - start

    usage = result.get("usage", {})
    timings = result.get("timings", {})

    prompt_tokens = usage.get("prompt_tokens", 0)
    completion_tokens = usage.get("completion_tokens", 0)

    prompt_speed = timings.get("prompt_per_second")
    decode_speed = timings.get("predicted_per_second")

    message = result.get("choices", [{}])[0].get("message", {})
    content = message.get("content", "")
    reasoning = message.get("reasoning_content", "")

    print("=" * 70)
    print("PrismML / llama-server API benchmark")
    print("=" * 70)

    print(f"Model: {MODEL}")
    print(f"Prompt tokens: {prompt_tokens}")
    print(f"Completion tokens: {completion_tokens}")
    print(f"Wall time: {wall:.2f}s")

    if prompt_speed is not None:
        print(f"Prompt processing: {prompt_speed:.2f} tok/s")

    if decode_speed is not None:
        print(f"Decode speed: {decode_speed:.2f} tok/s")

    if completion_tokens and not decode_speed:
        print(f"Approx completion speed: {completion_tokens / wall:.2f} tok/s")

    print(f"Finish reason: {result.get('choices', [{}])[0].get('finish_reason')}")

    if reasoning:
        print("\nReasoning tokens/content returned:")
        print(reasoning[:2000])

    print("\nFinal answer:")
    print(content)

    print("=" * 70)

    return result


result = run_api()


# Optional: expose the PrismML llama.cpp API through a temporary public URL

The tunnel points to **port 8080**. The server is OpenAI-compatible, so clients use `/v1`.


## 10. Install Cloudflare Tunnel

Skip this section if you only need the local Ollama API inside Kaggle.

In [ ]:
!curl -fL "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb" -o /tmp/cloudflared-linux-amd64.deb
!dpkg -i /tmp/cloudflared-linux-amd64.deb
!cloudflared --version

## 11. Start the Cloudflare Quick Tunnel

In [ ]:
import queue
import re
import subprocess
import threading
import time

def pipe_process_output(process, output_queue):
    for line in iter(process.stdout.readline, ""):
        output_queue.put(line)

tunnel_is_running = (
    "cloudflared_process" in globals()
    and cloudflared_process.poll() is None
    and "PUBLIC_LLAMA_URL" in globals()
    and PUBLIC_LLAMA_URL
)

if not tunnel_is_running:
    cloudflared_process = subprocess.Popen(
        [
            "cloudflared", "tunnel",
            "--no-autoupdate",
            "--protocol", "http2",
            "--edge-ip-version", "4",
            "--url", LLAMA_URL,
            "--http-host-header", "localhost:8080",
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    output_queue = queue.Queue()
    threading.Thread(
        target=pipe_process_output,
        args=(cloudflared_process, output_queue),
        daemon=True,
    ).start()

    deadline = time.monotonic() + 45
    PUBLIC_LLAMA_URL = None

    while time.monotonic() < deadline:
        if cloudflared_process.poll() is not None and output_queue.empty():
            break
        try:
            line = output_queue.get(timeout=1)
        except queue.Empty:
            continue
        match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
        if match:
            PUBLIC_LLAMA_URL = match.group(0)
            break

    if not PUBLIC_LLAMA_URL:
        raise RuntimeError("Cloudflare did not produce a public URL. Rerun this cell.")

PUBLIC_OLLAMA_URL = PUBLIC_LLAMA_URL  # compatibility alias

print("Public llama.cpp API:", PUBLIC_LLAMA_URL)
print("OpenAI-compatible base URL:", f"{PUBLIC_LLAMA_URL}/v1")
print("Model name:", MTP_MODEL)


## 11. Test the local or public OpenAI-compatible API

In [ ]:
import json
import urllib.request

def chat(prompt, base_url=LLAMA_URL, system=None):
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})

    payload = {
        "model": MTP_MODEL,
        "messages": messages,
        "stream": False,
        "temperature": 1.0,
        "top_p": 0.95,
        "top_k": 20,
        "min_p": 0.05,
        "max_tokens": 4096,
        "reasoning_effort": "medium",
    }

    request = urllib.request.Request(
        f"{base_url.rstrip('/')}/v1/chat/completions",
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(request, timeout=600) as response:
        result = json.loads(response.read())
    return result["choices"][0]["message"]["content"]

print(chat("Write a one-line Python hello-world program."))


### Test through the public tunnel

In [ ]:
print(chat("Hello from the Cloudflare tunnel.", base_url=PUBLIC_LLAMA_URL))

## Optional: OpenAI-compatible client settings

- **Base URL:** `https://YOUR-TUNNEL.trycloudflare.com/v1`
- **Model:** `ternary-bonsai-2-27b-pq2`
- **API key:** a placeholder value if your client requires one

The tunnel URL is temporary.


## Interactive Ollama CLI (optional)

Run this only after all setup cells are complete.

In [ ]:
if LLAMA_CLI_BIN:
    print("Launching PrismML llama-cli. Exit with Ctrl+C.")
    subprocess.run(
        [
            LLAMA_CLI_BIN,
            "-m", str(MODEL_PATH),
            "-ngl", "99",
            "-fa", "on",
            "-c", str(NUM_CTX),
            "--temp", "1.0",
            "--top-p", "0.95",
            "--top-k", "20",
            "--min-p", "0.05",
            "--jinja",
        ],
        check=False,
    )
else:
    print("PrismML llama-cli was not found.")


## 13. Full-context / memory diagnostics

The server is configured for the model's full **262,144-token** context. The 4-bit KV cache is specifically intended for very long contexts; it is a memory optimization, not a speed optimization. citeturn0search0turn0search6

For a single-user workload, `-np 1` is intentional: it lets the server maintain one unified conversation/cache rather than splitting a conversation across multiple slots.


In [ ]:
print("=== Bonsai 2 runtime ===")
print("Model:", MODEL_PATH)
print("Context:", NUM_CTX)
print("KV4:", BONSAI_KV4)
print("GPU layers:", BONSAI_NGL)
print("Server slots:", LLAMA_NP)
print("RAM prompt cache:", f"{LLAMA_CACHE_RAM} MB")
print("Server:", LLAMA_URL)
print()
print("=== GPU ===")
subprocess.run(["nvidia-smi"], check=False)
print()
print("=== Server log tail ===")
subprocess.run(["bash", "-lc", "tail -n 40 /tmp/bonsai-llama-server.log 2>/dev/null || true"], check=False)


In [ ]:
import json
import time
import urllib.request
import urllib.error

MODEL = "/kaggle/working/bonsai-model/Ternary-Bonsai-2-27B-PQ2_0.gguf"

# Safely below the 262,144-token limit.
TARGET_CHARS = 250_000 * 4

chunk = (
    "This is a direct long context test for Ternary Bonsai 2. "
    "The model must process and retain this context correctly. "
)

text = (chunk * ((TARGET_CHARS // len(chunk)) + 1))[:TARGET_CHARS]

prompt = text + """
END OF LONG CONTEXT.

Reply with exactly:
262K TEST PASSED
"""

payload = {
    "model": MODEL,
    "messages": [
        {"role": "user", "content": prompt}
    ],
    "stream": False,
    "temperature": 0.0,
    "max_tokens": 32,
}

req = urllib.request.Request(
    f"{LLAMA_URL}/v1/chat/completions",
    data=json.dumps(payload).encode(),
    headers={"Content-Type": "application/json"},
    method="POST",
)

print("Sending ~250K-token context...")
start = time.perf_counter()

try:
    with urllib.request.urlopen(req, timeout=3600) as r:
        result = json.loads(r.read())
except urllib.error.HTTPError as e:
    print(f"HTTP {e.code}: {e.reason}")
    print(e.read().decode(errors="replace"))
    raise

elapsed = time.perf_counter() - start

usage = result.get("usage", {})
timings = result.get("timings", {})
choice = result["choices"][0]
message = choice["message"]

print("\n" + "=" * 60)
print("LONG-CONTEXT TEST")
print("=" * 60)
print(f"Wall time:       {elapsed:.2f}s")
print(f"Prompt tokens:   {usage.get('prompt_tokens'):,}")
print(f"Output tokens:   {usage.get('completion_tokens'):,}")
print(f"Total tokens:    {usage.get('total_tokens'):,}")
print(f"Finish reason:   {choice.get('finish_reason')}")

if timings.get("prompt_per_second"):
    print(f"Prompt speed:    {timings['prompt_per_second']:.2f} tok/s")

if timings.get("predicted_per_second"):
    print(f"Decode speed:    {timings['predicted_per_second']:.2f} tok/s")

print("\nResponse:")
print(message.get("content", ""))

print("\n✅ Context request accepted.")

In [ ]:

import json
import time
import urllib.request
import urllib.error

MODEL = "/kaggle/working/bonsai-model/Ternary-Bonsai-2-27B-PQ2_0.gguf"

# Recreate the SAME ~217K context used in the previous test.
TARGET_CHARS = 250_000 * 4

chunk = (
    "This is a direct long context test for Ternary Bonsai 2. "
    "The model must process and retain this context correctly. "
)

text = (chunk * ((TARGET_CHARS // len(chunk)) + 1))[:TARGET_CHARS]

prompt = text + """
END OF LONG CONTEXT.

Now answer this question in about 100 words:

Why are GPUs useful for running large language models?
"""

payload = {
    "model": MODEL,
    "messages": [
        {"role": "user", "content": prompt}
    ],
    "stream": False,
    "temperature": 0.2,
    "top_p": 0.9,

    # Large enough to allow reasoning + final answer.
    "max_tokens": 512,
}

req = urllib.request.Request(
    f"{LLAMA_URL}/v1/chat/completions",
    data=json.dumps(payload).encode(),
    headers={"Content-Type": "application/json"},
    method="POST",
)

print("Sending the same long context again...")
print("Prompt-cache reuse should make this much faster if the prefix matches.")
print()

start = time.perf_counter()

try:
    with urllib.request.urlopen(req, timeout=3600) as r:
        result = json.loads(r.read())

except urllib.error.HTTPError as e:
    print(f"HTTP {e.code}: {e.reason}")
    print(e.read().decode(errors="replace"))
    raise

elapsed = time.perf_counter() - start

usage = result.get("usage", {})
timings = result.get("timings", {})
choice = result["choices"][0]
message = choice["message"]

print("=" * 70)
print("217K CONTEXT + NORMAL GENERATION")
print("=" * 70)

print(f"Wall time:       {elapsed:.2f}s")
print(f"Prompt tokens:   {usage.get('prompt_tokens', 0):,}")
print(f"Output tokens:   {usage.get('completion_tokens', 0):,}")
print(f"Total tokens:    {usage.get('total_tokens', 0):,}")
print(f"Finish reason:   {choice.get('finish_reason')}")

if timings.get("cache_n") is not None:
    print(f"Cached tokens:   {timings.get('cache_n'):,}")

if timings.get("prompt_per_second") is not None:
    print(f"Prompt speed:    {timings['prompt_per_second']:.2f} tok/s")

if timings.get("predicted_per_second") is not None:
    print(f"Decode speed:    {timings['predicted_per_second']:.2f} tok/s")

if timings.get("predicted_ms") is not None:
    print(f"Decode time:     {timings['predicted_ms'] / 1000:.2f}s")

print("\n--- MODEL OUTPUT ---")
print(message.get("content", ""))

print("\n" + "=" * 70)
